**PLOTTING PREDICTIONS DISTRIBUTION**



In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import plotly.io as pio
pio.renderers.default='notebook'
import plotly.express as px
import datetime
import ast
import subprocess
import matplotlib.colors as mcolors
import seaborn as sns
import plotly.express as px

In [2]:
COLLOR_PALLET = {
            'Other human': '#2986cc', # BLUE
            'Electro-mechanical': '#cc0000', # RED
            'Voice': '#6aa84f', #  green 6aa84f
            'Motorised transport': '#ffa500', # orange
            'Geonature': '#8e7cc3', # PURPLE
            'Animal': '#9b5f00', # BROWN
            'Music': '#d172a4', # PINK
            'Background': '#000000', # BLACK
            'Other Sounds': '#c9d631', # yellow
            'Social/communal': '#d8cbf8', # Light purple
            'Human movement': '#40b674', # light green 40b674
        }

def extract_location(file_path):
    file_name = os.path.basename(file_path)
    # print(f"File name: {file_name}")
    # Split the file name by the underscore
    file_name = file_name.split("_")[2]
    # print(f"File name after split: {file_name}")
    return file_name
    
def postprocessing_df(df_preds):
    # df_preds.drop(columns=['classes_custom', 'probabilities_custom', 'sum_probs_custom', 'sum_probs_original'],inplace=True)
    # df_preds.rename(columns={'datetime':'filenames', 'classes_original':'classes'}, inplace=True)
    df_preds['class'] = df_preds['class'].apply(ast.literal_eval)
    
    return df_preds

def insert_time(df_preds):
    df_preds['date'] = pd.to_datetime(df_preds['date'])

    df_preds['year'] = df_preds.apply(lambda x: x['date'].year, axis= 1)
    df_preds['month'] = df_preds.apply(lambda x: x['date'].month, axis= 1)
    df_preds['day'] = df_preds.apply(lambda x: x['date'].day, axis= 1)

    df_preds['day_name'] = df_preds.apply(lambda x: x['date'].day_name(), axis= 1)
    df_preds['weekday'] = df_preds.apply(lambda x: x['date'].weekday(), axis= 1)
    
    df_preds['hour'] = df_preds.apply(lambda x: x['date'].hour, axis= 1)
    df_preds['hour_min'] = df_preds.apply(lambda x: str(x['date'].hour) + '_' + str(x['date'].minute),axis=1)
    
    df_preds['time'] = df_preds.apply(lambda x: x['date'].time(),axis=1)
    
    return df_preds

def index_and_explode(df):
    df.set_index('date',inplace=True)
    df = df.explode('class')
    df['display_name'] = df['class']
    df['number'] = 1
    
    return df

def merge_dataframes(df, union):
    df = df.merge(union,how='left',on='display_name')
    
    return df

def remove_label(df, column, label):
    df_no_label = df[df[column] != label]
    return df_no_label, label

# sort by time column called datetime
def sort_df(df):
    df.sort_values(by=['date'], inplace=True)
    return df

def output_dir(path: str):
    # get the abs path and remove the last element
    path = os.path.abspath(path).split("\\")[:-2]
    path = "/".join(path)
    
    visualization_dir = path + "/Visualizations"
    os.makedirs(visualization_dir, exist_ok=True)
    
    return visualization_dir

# get the last git tag version
def list_git_tags():
    try:
        tags = tags = subprocess.check_output(["git", "tag"]).strip().decode()
        return tags.split('\n')
    except subprocess.CalledProcessError:
        return None
    
def select_tag(tags):
    for i, tag in enumerate(tags):
        print(f"{i}: {tag}")
    choice = int(input("Select the tag to use: "))
    tag_selected = tags[choice]
    # replace "." with "_" to be able to use it as a file name
    tag_selected = tag_selected.replace(".", "_")
        
    return tag_selected

def get_stable_version():
    tags = list_git_tags()
    # get the latest stable version
    tag_selected = tags[-1]
    print(f"Latest stable version with period: {tag_selected}")
    # replace "." with "_" to be able to use it as a file name
    tag_selected = tag_selected.replace(".", "_")
    
    print(f"Latest stable version with underscore: {tag_selected}")
    
    return tag_selected

stable_version = get_stable_version()

Latest stable version with period: v1.1
Latest stable version with underscore: v1_1


In [3]:
# df_preds = pd.read_csv(f'avg_predictions_{abrev}.csv', converters={'classes': eval})
# yammnet_class_map = 'yamnet_class_map_rev_FINAL.csv'
yammnet_class_map = r"C:\Users\scjaa\AAC - CENTRO DE ACUSTICA APLICADA, S.L\I + D + i - Documentos\Modelos_IA\AAC_IA_Puerto\yamnet_class_AAC_v2_0.csv"
# yammnet_class_map = r"C:\Users\scjaa\Documents\GitHubRepos\AAC\AI_Model\Urban_Model\taxonomy_mapping\yamnet_class_AAC_v1_0.csv"

union = pd.read_csv(yammnet_class_map,sep=';')

path_input = input("Enter the path to the csv file: ")
path_input = path_input.replace('"', '')

# make visualization directory
visualization_dir = output_dir(path_input)

# get the location
title = extract_location(path_input)

# get the csv
df_preds = pd.read_csv(path_input)

df_preds

IndexError: list index out of range

In [64]:
df_preds = sort_df(df_preds)
# df_preds['class'] = df_preds['class'].apply(ast.literal_eval)
df_preds['class'] = df_preds['class'].apply(lambda x: x.split(', '))
df_preds = insert_time(df_preds)
df_preds

,filename,date,class,probability,year,month,day,day_name,weekday,hour,hour_min,time
0,20231205_111755.WAV,2023-12-05 11:17:55,"[Speech, Inside, small room, Biting, Animal, C...","0.3267, 0.1010, 0.0691, 0.0567, 0.0460",2023,12,5,Tuesday,1,11,11_17,11:17:55
1,20231205_111825.WAV,2023-12-05 11:18:25,"[Speech, Vehicle, Inside, small room, Motor ve...","0.2599, 0.0853, 0.0427, 0.0241, 0.0227",2023,12,5,Tuesday,1,11,11_18,11:18:25
2,20231205_113330.WAV,2023-12-05 11:33:30,"[Speech, Vehicle, Motor vehicle (road), Outsid...","0.1653, 0.1342, 0.0395, 0.0353, 0.0338",2023,12,5,Tuesday,1,11,11_33,11:33:30
3,20231205_114835.WAV,2023-12-05 11:48:35,"[Vehicle, Speech, Motor vehicle (road), Outsid...","0.1817, 0.1011, 0.0498, 0.0400, 0.0383",2023,12,5,Tuesday,1,11,11_48,11:48:35
4,20231205_120340.WAV,2023-12-05 12:03:40,"[Vehicle, Speech, Motor vehicle (road), Car, B...","0.1983, 0.0681, 0.0613, 0.0539, 0.0372",2023,12,5,Tuesday,1,12,12_3,12:03:40
...,...,...,...,...,...,...,...,...,...,...,...,...
573,20231211_101805.WAV,2023-12-11 10:18:05,"[Vehicle, Speech, Inside, small room, Motor ve...","0.1274, 0.0908, 0.0663, 0.0450, 0.0399",2023,12,11,Monday,0,10,10_18,10:18:05
574,20231211_103310.WAV,2023-12-11 10:33:10,"[Vehicle, Speech, Inside, small room, Motor ve...","0.1220, 0.1202, 0.0499, 0.0446, 0.0407",2023,12,11,Monday,0,10,10_33,10:33:10
575,20231211_104815.WAV,2023-12-11 10:48:15,"[Speech, Vehicle, Motor vehicle (road), Inside...","0.1419, 0.1223, 0.0450, 0.0416, 0.0395",2023,12,11,Monday,0,10,10_48,10:48:15
576,20231211_110320.WAV,2023-12-11 11:03:20,"[Vehicle, Speech, Inside, small room, Motor ve...","0.1264, 0.0939, 0.0655, 0.0423, 0.0407",2023,12,11,Monday,0,11,11_3,11:03:20


In [66]:
df = df_preds.copy()
df = index_and_explode(df)
df

,filename,class,probability,year,month,day,day_name,weekday,hour,hour_min,time,display_name,number
date,,,,,,,,,,,,,
2023-12-05 11:17:55,20231205_111755.WAV,Speech,"0.3267, 0.1010, 0.0691, 0.0567, 0.0460",2023,12,5,Tuesday,1,11,11_17,11:17:55,Speech,1
2023-12-05 11:17:55,20231205_111755.WAV,Inside,"0.3267, 0.1010, 0.0691, 0.0567, 0.0460",2023,12,5,Tuesday,1,11,11_17,11:17:55,Inside,1
2023-12-05 11:17:55,20231205_111755.WAV,small room,"0.3267, 0.1010, 0.0691, 0.0567, 0.0460",2023,12,5,Tuesday,1,11,11_17,11:17:55,small room,1
2023-12-05 11:17:55,20231205_111755.WAV,Biting,"0.3267, 0.1010, 0.0691, 0.0567, 0.0460",2023,12,5,Tuesday,1,11,11_17,11:17:55,Biting,1
2023-12-05 11:17:55,20231205_111755.WAV,Animal,"0.3267, 0.1010, 0.0691, 0.0567, 0.0460",2023,12,5,Tuesday,1,11,11_17,11:17:55,Animal,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-11 11:18:25,20231211_111825.WAV,Vehicle,"0.1815, 0.1098, 0.0550, 0.0397, 0.0389",2023,12,11,Monday,0,11,11_18,11:18:25,Vehicle,1
2023-12-11 11:18:25,20231211_111825.WAV,Inside,"0.1815, 0.1098, 0.0550, 0.0397, 0.0389",2023,12,11,Monday,0,11,11_18,11:18:25,Inside,1
2023-12-11 11:18:25,20231211_111825.WAV,small room,"0.1815, 0.1098, 0.0550, 0.0397, 0.0389",2023,12,11,Monday,0,11,11_18,11:18:25,small room,1


In [67]:
df = merge_dataframes(df, union)
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df

,filename,class,probability,year,month,day,day_name,weekday,hour,hour_min,time,display_name,number,index,mid,iso_taxonomy,Brown_Level_1,Brown_Level_2,Brown_Level_3,NoisePort
0,20231205_111755.WAV,Speech,"0.3267, 0.1010, 0.0691, 0.0567, 0.0460",2023,12,5,Tuesday,1,11,11_17,11:17:55,Speech,1,0.0,/m/09x0r,Voice,NaN,Voice,speech,Human
1,20231205_111755.WAV,Inside,"0.3267, 0.1010, 0.0691, 0.0567, 0.0460",2023,12,5,Tuesday,1,11,11_17,11:17:55,Inside,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20231205_111755.WAV,small room,"0.3267, 0.1010, 0.0691, 0.0567, 0.0460",2023,12,5,Tuesday,1,11,11_17,11:17:55,small room,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20231205_111755.WAV,Biting,"0.3267, 0.1010, 0.0691, 0.0567, 0.0460",2023,12,5,Tuesday,1,11,11_17,11:17:55,Biting,1,50.0,/m/07pdhp0,Other human,NaN,Other human,Other human,Human
4,20231205_111755.WAV,Animal,"0.3267, 0.1010, 0.0691, 0.0567, 0.0460",2023,12,5,Tuesday,1,11,11_17,11:17:55,Animal,1,67.0,/m/0jbk,Nature,NaN,Animal,Animal,Animal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3865,20231211_111825.WAV,Vehicle,"0.1815, 0.1098, 0.0550, 0.0397, 0.0389",2023,12,11,Monday,0,11,11_18,11:18:25,Vehicle,1,294.0,/m/07yv9,Roadway traffic,NaN,Motorised transport,roadway traffic,Heavy Vehicle
3866,20231211_111825.WAV,Inside,"0.1815, 0.1098, 0.0550, 0.0397, 0.0389",2023,12,11,Monday,0,11,11_18,11:18:25,Inside,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3867,20231211_111825.WAV,small room,"0.1815, 0.1098, 0.0550, 0.0397, 0.0389",2023,12,11,Monday,0,11,11_18,11:18:25,small room,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3868,20231211_111825.WAV,Motor vehicle (road),"0.1815, 0.1098, 0.0550, 0.0397, 0.0389",2023,12,11,Monday,0,11,11_18,11:18:25,Motor vehicle (road),1,300.0,/m/012f08,Roadway traffic,NaN,Motorised transport,roadway traffic,Other Engine


In [69]:
# start_date = df.sort_values('date')['date'].iloc[0]
# end_date = df.sort_values('date')['date'].iloc[-1]
# print('Inicio Monitoreo: {} \nFinal Monitoreo: {}'.format(start_date,end_date))

unique_classes = set(df['class'].unique())
df_no_Nature, label = remove_label(df, 'Brown_Level_2', 'Nature')

## **PLOTTING GLOBAL DASH BOARD**

In [70]:
# display_name, Brown_Level_1, Brown_Level_2, Brown_Level_3
display_name = 'display_name'
brown_2 = 'Brown_Level_2'
brown_3 = 'Brown_Level_3'
port = 'NoisePort'

# set what detail level to use
deepth = brown_2

In [76]:
# remove all the Unnamed columns+
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
# remove nana values for the selected column
df = df[df[deepth].notna()]

In [83]:
fig = px.treemap(df, 
                 path=[deepth, 'class'], 
                 values='number',
                 color=deepth,  #for coloring
                 color_discrete_map=COLLOR_PALLET 
                )

fig.update_layout(title=f'{title} Global | Urban Model | From {df["time"].iloc[0]} to {df["time"].iloc[-1]}')
fig.show()

fig.write_html(f"{visualization_dir}/{title}_Global_UrbanModel_{stable_version}.html")
print(f"File saved at: {visualization_dir}/{title}_Global_UrbanModel_{stable_version}.html")

File saved at: //192.168.205.117/AAC_Server/OCIO/OCIO_BILBAO/CAMPAÑA_5/5-Resultados/P1_FA5AAD/AI_MODEL/Visualizations/P1_Global_UrbanModel_v1_1.html


In [ ]:
# fig = px.treemap(df_no_Nature, 
#                     path=['Brown_Level_2', 'classes'], 
#                     values='number'
#                 )
# fig.update_layout(title=f'{title} Global Sin {label} | {df["date"].iloc[0]} - {df["date"].iloc[-1]}')

# # save figure
# fig.write_html(f"{visualization_dir}/{title}_Global_Sin_{label}_{stable_version}.html")

# fig.show()


**PLOTTING EACH DAY**

In [79]:
for day in df['day'].unique():
    day_df = df[df['day'] == day]
    fig = px.treemap(day_df, 
                     path=[deepth, 'class'], 
                     values='number',
                     color=deepth,
                     color_discrete_map=COLLOR_PALLET
                    )
    fig.update_layout(title=f'{title} | Urban Model | {day_df["year"].iloc[0]}-{day_df["month"].iloc[0]}-{day} - {day_df["day_name"].iloc[0]}')
    
    # save figure
    fig.write_html(f"{visualization_dir}/{title}_Day_{day}_Global_UrbanModel_{stable_version}.html")
    fig.show()

# PLOTTING FOR EACH 4 HOURS STARTING AT 07:00 AM

In [ ]:
for column in df.columns:
    if 'Unnamed' in column:
        df.drop(columns=column, inplace=True)

In [ ]:
# set the datetime column as index
df.set_index('filenames', inplace=True)

In [ ]:
# for each 4-hour interval starting at 07:00 am
for day in df['day'].unique():
    day_df = df[df['day'] == day]
    print(f"Day: {day}")
    # print hour
    print(f"Hour: {day_df['hour'].iloc[0]}")

    for hour in range(7, 24, 4):  # 7, 11, 15, 19, 23
        # Select data between 'hour' and 'hour + 4', wrapping around midnight if necessary
        next_hour = hour + 4
        print(f"Hour: {hour}")
        print(f"Next hour: {next_hour}")
        
        if next_hour > 23:
            hour_df = df[(df['hour'] >= hour) | (df['hour'] < next_hour - 24)]
        else:
            hour_df = df[(df['hour'] >= hour) & (df['hour'] < next_hour)]

        if not hour_df.empty:
            print(f"Time Interval: {hour}:00 to {next_hour}:00")

            fig = px.treemap(hour_df, 
                            path=[deepth, 'classes'], 
                            values='number',
                            color=deepth,
                            color_discrete_map=COLLOR_PALLET
                            )
            fig.update_layout(title=f'{title} | Urban Model | {hour_df["year"].iloc[0]}-{hour_df["month"].iloc[0]}-{hour_df["day"].iloc[0]} - {hour_df["day_name"].iloc[0]} - {hour}:00 to {next_hour}:00')
            
            fig.show()

Day: 22
Hour: 9
Hour: 7
Next hour: 11
Time Interval: 7:00 to 11:00


Hour: 11
Next hour: 15
Time Interval: 11:00 to 15:00


Hour: 15
Next hour: 19
Time Interval: 15:00 to 19:00


Hour: 19
Next hour: 23
Time Interval: 19:00 to 23:00


Hour: 23
Next hour: 27
Time Interval: 23:00 to 27:00


...

Day: 27
Hour: 0
Hour: 7
Next hour: 11
Time Interval: 7:00 to 11:00


Hour: 11
Next hour: 15
Time Interval: 11:00 to 15:00


Hour: 15
Next hour: 19
Time Interval: 15:00 to 19:00


Hour: 19
Next hour: 23
Time Interval: 19:00 to 23:00


Hour: 23
Next hour: 27
Time Interval: 23:00 to 27:00


In [ ]:
for day in df_no_Nature['day'].unique():
    day_df = df_no_Nature[df_no_Nature['day'] == day]
    fig = px.treemap(df_no_Nature, 
                     path=['Brown_Level_2', 'classes'], 
                     values='number'
                    )
    fig.update_layout(title=f'{title} Sin {label} | {day_df["year"].iloc[0]}-{day_df["month"].iloc[0]}-{day}')

    fig.show()

**PLOTTING SPECIFIC TIME SPANS**

In [ ]:
START_TIME_SPECIFIC = "2022-10-29 13:00:00"
END_TIME_SPECIFIC = "2022-10-29 15:15:00"

In [ ]:
start_time = pd.Timestamp(START_TIME_SPECIFIC)
end_time = pd.Timestamp(END_TIME_SPECIFIC)

filtered_df = df[(df['date'] >= start_time) & (df['date'] <= end_time)]

filtered_df['number'] = 1
fig = px.treemap(filtered_df, 
                    path=['Brown_Level_2', 'classes'], 
                    values='number',
                    color='Brown_Level_2',
                    color_discrete_map=COLLOR_PALLET
                )
fig.update_layout(title=f'{title} | {start_time} - {end_time} ')

# save fuiture
fig.write_html(f"{visualization_dir}/{title}_{START_TIME_SPECIFIC}_{END_TIME_SPECIFIC}_{stable_version}.html")

fig.show()

OSError: [Errno 22] Invalid argument: '\\\\192.168.205.117\\AAC_Server\\INDUSTRIA\\23603_GRANTIERRA\\5-Resultados\\Audiomoth2\\URBAN_Model\\Visualizations\\GRANTIERRA-P2_2022-10-29 13:00:00_2022-10-29 15:15:00_v1_0.html'

In [ ]:
start_time = pd.Timestamp(START_TIME_SPECIFIC)
end_time = pd.Timestamp(END_TIME_SPECIFIC)
filtered_df = df_no_Nature[(df['date'] >= start_time) & (df['date'] <= end_time)]

filtered_df['number'] = 1
fig = px.treemap(filtered_df, 
                    path=['Brown_Level_2', 'classes'], 
                    values='number',
                    color='Brown_Level_2',
                    color_discrete_map=COLLOR_PALLET
                )
fig.update_layout(title=f'{title} Sin {label} | {start_time} - {end_time} ')

# save figure
fig.write_html(f"{visualization_dir}/{title}_{START_TIME_SPECIFIC}_{END_TIME_SPECIFIC}_Sin_{label}.html")

fig.show()

/tmp/ipykernel_16501/2334339269.py:3: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_16501/2334339269.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
start_time = pd.Timestamp("2022-10-29 20:00:00")
end_time = pd.Timestamp("2022-10-29 21:00:00")
filtered_two_df = df[(df['date'] >= start_time) & (df['date'] <= end_time)]

filtered_df = df[(df['date'] >= start_time) & (df['date'] <= end_time)]

filtered_df['number'] = 1
fig = px.treemap(filtered_df, 
                    path=['Brown_Level_2', 'classes'], 
                    values='number',
                    color='Brown_Level_2',
                    color_discrete_map=COLLOR_PALLET
                )
fig.update_layout(title=f'{title} | {start_time} - {end_time} ')
fig.show()

/tmp/ipykernel_16501/3784541672.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
start_time = pd.Timestamp("2022-10-20 13:30:00")
end_time = pd.Timestamp("2022-10-20 14:50:00")
filtered_two_df = df_no_Nature[(df['date'] >= start_time) & (df['date'] <= end_time)]

filtered_two_df['number'] = 1
fig = px.treemap(filtered_two_df, 
                    path=['Brown_Level_2', 'classes'], 
                    values='number',
                    color='Brown_Level_2',
                    color_discrete_map=COLLOR_PALLET
                )
fig.update_layout(title=f'{title} Sin {label} | {start_time} - {end_time}')
fig.show()

/tmp/ipykernel_16501/2926250780.py:3: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



In [ ]:
start_time = pd.Timestamp("2022-10-29 21:20:00")
end_time = pd.Timestamp("2022-10-29 23:55:00")
filtered_df = df[(df['date'] >= start_time) & (df['date'] <= end_time)]

filtered_df['number'] = 1
fig = px.treemap(filtered_df, 
                    path=['Brown_Level_2', 'classes'], 
                    values='number',
                    color='Brown_Level_2',
                    color_discrete_map=COLLOR_PALLET
                )
fig.update_layout(title=f'{title} | {start_time} - {end_time} ')
fig.show()

/tmp/ipykernel_16501/245140038.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
start_time = pd.Timestamp("2022-11-09 09:45:00")
end_time = pd.Timestamp("2022-11-09 12:00:00")
filtered_df = df[(df['date'] >= start_time) & (df['date'] <= end_time)]

filtered_df['number'] = 1
fig = px.treemap(filtered_df, 
                    path=['Brown_Level_2', 'classes'], 
                    values='number',
                    color='Brown_Level_2',
                    color_discrete_map=COLLOR_PALLET
                )
fig.update_layout(title=f'{title} | {start_time} - {end_time} ')
fig.show()

/tmp/ipykernel_16501/3523570605.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
start_time = pd.Timestamp("2022-11-14 11:00:00")
end_time = pd.Timestamp("2022-11-14 13:00:00")
filtered_df = df[(df['date'] >= start_time) & (df['date'] <= end_time)]

filtered_df['number'] = 1
fig = px.treemap(filtered_df, 
                    path=['Brown_Level_2', 'classes'], 
                    values='number',
                    color='Brown_Level_2',
                    color_discrete_map=COLLOR_PALLET
                )
fig.update_layout(title=f'{title} | {start_time} - {end_time} ')
fig.show()

/tmp/ipykernel_16501/408549486.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

